# 06 — Audit and Replay

**Notebook 6 of the *Developer Guide to Disciplined Trading* series.** The closer.

> Prerequisites: [`01`](./01-foundations-techtrade-and-analysis.ipynb), [`02`](./02-morning-scan.ipynb), [`03`](./03-single-position-deep-dive.ipynb), [`04`](./04-validation-gate.ipynb), [`05`](./05-tuning-per-sector.ipynb).

---

## The question this notebook answers

> *"The morning scan ran. I took some trades. **Did I follow my own discipline, and do the trade outcomes match what the engine said?**"*

This is end-of-day discipline. The morning notebook (02) produced an Excel workbook of recommendations. Notebook 03 zoomed in on one. Notebook 04 validated it. Notebook 05 maybe tuned the sector. This notebook **closes the loop**:

1. Load yesterday's (or any past) `techtrade_<date>.xlsx` from `Analysis/exports/`.
2. Replay each plan against the **actual forward bars** that have happened since.
3. Compare engine prediction vs realized outcome — Was the recommendation right? Where did stops hit before targets? Where did Alex deviate from the recommendation?
4. Aggregate into a **journal entry**: total signals, taken/skipped, win rate, average R captured, deviations.

This is the disciplined trader's evening routine — done daily, it's the single most powerful behavior-change feedback loop. The engine doesn't grade Alex; **Alex grades himself by replaying the engine.**

## What Alex takes away

- A **discipline ledger** for the period: hit rate, R captured per trade, where conviction-vs-outcome agreed/disagreed.
- The **deviation log**: every plan he should have taken but skipped, and every plan he should have skipped but took.
- A **journal entry** (one markdown block per day) suitable for pasting into his trading journal.

## Wall-clock

**2-10 minutes** depending on how many plans the past workbook contained. All the heavy lifting (fmp_cached fetches for the replay window) is the slow part.

## 1. Setup + find a workbook to audit

In [ ]:
from pathlib import Path
from datetime import date, timedelta

EXPORTS_DIR = Path.cwd().parent / "Analysis" / "exports"
# If the notebook is in notebooks/, the exports dir is at <repo>/Analysis/exports/.
# Walk up to find it if the relative path doesn't resolve.
if not EXPORTS_DIR.is_dir():
    cursor = Path.cwd().resolve()
    while cursor != cursor.parent:
        candidate = cursor / "Analysis" / "exports"
        if candidate.is_dir():
            EXPORTS_DIR = candidate
            break
        cursor = cursor.parent

print(f"Looking for workbooks in: {EXPORTS_DIR}")

if not EXPORTS_DIR.is_dir():
    raise RuntimeError(
        f"No exports directory found. Run notebook 02's export cell first to produce a workbook."
    )

xlsx_files = sorted(EXPORTS_DIR.glob("techtrade_*.xlsx"))
print(f"Found {len(xlsx_files)} workbook(s):")
for f in xlsx_files[-5:]:
    age = (date.today() - date.fromtimestamp(f.stat().st_mtime)).days
    print(f"  {f.name}  (modified {age}d ago)")

In [ ]:
if not xlsx_files:
    raise RuntimeError("No workbooks to audit. Run notebook 02 first.")

# Default: audit the most recent workbook that's at least a few days old
# (need forward bars to replay against).
WORKBOOK = xlsx_files[-1]
print(f"Auditing: {WORKBOOK}")
print(f"Workbook date: {date.fromtimestamp(WORKBOOK.stat().st_mtime).isoformat()}")

## 2. Load the recommendations

The workbook's `Recommendations` sheet has one row per plan. Strip out the disclaimer row at the top, then read the rest into a DataFrame.

In [ ]:
import pandas as pd
import openpyxl

wb = openpyxl.load_workbook(WORKBOOK, read_only=True, data_only=True)
print(f"Sheets ({len(wb.sheetnames)}): {wb.sheetnames}")

# Find the header row in Recommendations (skip the disclaimer at the top).
ws = wb["Recommendations"]
rows = list(ws.iter_rows(values_only=True))
wb.close()

# Find the first row that looks like a header (has 'symbol' in it).
header_idx = next((i for i, r in enumerate(rows) if r and "symbol" in [str(c).lower() if c else "" for c in r]), None)
if header_idx is None:
    raise RuntimeError("Could not find header row in Recommendations sheet.")

header = [str(c) if c is not None else "" for c in rows[header_idx]]
data_rows = [r for r in rows[header_idx + 1:] if r and r[0] is not None]

recs_df = pd.DataFrame(data_rows, columns=header)
print(f"\nLoaded {len(recs_df)} recommendations.")
recs_df.head()

## 3. Replay against actual forward bars

For each recommendation in the workbook, fetch the actual OHLCV bars that have happened since the recommendation's `as_of` date. Then determine:

- **Did the entry trigger?** (Did price reach the entry price?)
- **Which exit hit first?** (Stop, target, or time stop?)
- **Realized P&L?** (Difference between fill prices × quantity, minus assumed slippage + commission)

This is the replay engine. It uses `obb.equity.price.historical` for the bar window and applies a simple no-look-ahead fill rule.

In [ ]:
from openbb import obb
from decimal import Decimal
from datetime import datetime, timedelta

def fetch_forward(symbol: str, as_of_date: date, days_ahead: int = 30) -> pd.DataFrame | None:
    """Fetch the next `days_ahead` business days of OHLCV after `as_of_date`."""
    start = as_of_date + timedelta(days=1)
    end = start + timedelta(days=days_ahead * 2)  # business days != calendar; pad generously
    try:
        df = obb.equity.price.historical(
            symbol=symbol,
            start_date=str(start),
            end_date=str(end),
            provider="fmp_cached",
        ).to_dataframe()
        # Drop any bars at or before as_of (no look-ahead).
        return df[df.index.date > as_of_date]
    except Exception as exc:
        print(f"  fetch failed for {symbol}: {exc}")
        return None

def replay_one(rec: pd.Series, bars: pd.DataFrame) -> dict:
    """Replay one recommendation against its forward bars."""
    if bars is None or bars.empty:
        return {"status": "no-forward-bars"}
    entry = float(rec["entry_price"]) if rec.get("entry_price") else None
    stop  = float(rec["stop_price"])  if rec.get("stop_price")  else None
    targ  = float(rec["target_price"]) if rec.get("target_price") else None
    qty   = float(rec["position_size"]) if rec.get("position_size") else 0.0
    action = rec.get("action", "")

    if entry is None or stop is None or targ is None or qty <= 0:
        return {"status": "missing-levels"}

    # T+1 entry: first bar's open is the fill (next-bar-open discipline).
    entry_bar = bars.iloc[0]
    entry_fill = float(entry_bar["open"])

    # Walk forward; first leg to hit wins.
    is_long = action == "BUY"
    for ts, bar in bars.iterrows():
        hi, lo = float(bar["high"]), float(bar["low"])
        if is_long:
            if lo <= stop:
                return {"status": "stop", "exit_price": stop, "exit_ts": ts.isoformat(),
                        "realized": (stop - entry_fill) * qty, "bars_held": (ts.date() - entry_bar.name.date()).days}
            if hi >= targ:
                return {"status": "target", "exit_price": targ, "exit_ts": ts.isoformat(),
                        "realized": (targ - entry_fill) * qty, "bars_held": (ts.date() - entry_bar.name.date()).days}
        else:  # SELL_SHORT
            if hi >= stop:
                return {"status": "stop", "exit_price": stop, "exit_ts": ts.isoformat(),
                        "realized": (entry_fill - stop) * qty, "bars_held": (ts.date() - entry_bar.name.date()).days}
            if lo <= targ:
                return {"status": "target", "exit_price": targ, "exit_ts": ts.isoformat(),
                        "realized": (entry_fill - targ) * qty, "bars_held": (ts.date() - entry_bar.name.date()).days}
    # Neither stop nor target hit; mark-to-market the last bar.
    final = float(bars.iloc[-1]["close"])
    realized = (final - entry_fill) * qty if is_long else (entry_fill - final) * qty
    return {"status": "open", "exit_price": final, "exit_ts": bars.index[-1].isoformat(),
            "realized": realized, "bars_held": len(bars)}

print("Replay helpers loaded.")

In [ ]:
# Run the replay over every recommendation in the workbook.
# Wall-clock: ~5-30s per recommendation depending on cache state.
results = []
workbook_date = date.fromtimestamp(WORKBOOK.stat().st_mtime)

for _, rec in recs_df.iterrows():
    sym = str(rec.get("symbol", "")).strip()
    if not sym:
        continue
    print(f"  replaying {sym} ...", end=" ")
    bars = fetch_forward(sym, workbook_date, days_ahead=30)
    outcome = replay_one(rec, bars)
    outcome.update({
        "symbol": sym,
        "as_of": workbook_date.isoformat(),
        "action": rec.get("action", ""),
        "conviction": rec.get("conviction", ""),
        "r:r": rec.get("risk_reward", None),
        "entry": rec.get("entry_price", None),
    })
    results.append(outcome)
    print(f"{outcome.get('status', '?'):<10} realized={outcome.get('realized', 0.0):+.2f}")

replay_df = pd.DataFrame(results)
print(f"\nReplayed {len(replay_df)} plans.")

## 4. Summary statistics

Aggregate the replay into the discipline metrics Alex cares about: hit rate, average R captured, breakdown by conviction tier.

In [ ]:
if replay_df.empty:
    print("No replay results to summarize.")
else:
    closed = replay_df[replay_df["status"].isin(["stop", "target"])]
    open_pos = replay_df[replay_df["status"] == "open"]
    failed = replay_df[~replay_df["status"].isin(["stop", "target", "open"])]

    wins = closed[closed["status"] == "target"]
    losses = closed[closed["status"] == "stop"]
    win_rate = len(wins) / len(closed) if len(closed) else 0.0
    avg_win_r = wins["r:r"].astype(float).mean() if len(wins) else 0.0
    avg_loss_r = -1.0  # by construction: a stop hit = -1R
    expectancy = win_rate * avg_win_r + (1 - win_rate) * avg_loss_r
    total_realized = replay_df["realized"].sum() if "realized" in replay_df.columns else 0.0

    print(f"--- Discipline summary for workbook {WORKBOOK.name} ---\n")
    print(f"  Total plans:           {len(replay_df)}")
    print(f"  Closed (stop/target):  {len(closed)}")
    print(f"  Still open:            {len(open_pos)}")
    print(f"  Replay failed:         {len(failed)}")
    print(f"\n  Wins (target hit):     {len(wins)}")
    print(f"  Losses (stop hit):     {len(losses)}")
    print(f"  Win rate:              {win_rate:.1%}")
    print(f"  Avg win R:             {avg_win_r:+.2f}")
    print(f"  Avg loss R:            {avg_loss_r:+.2f}")
    print(f"  Expectancy (per trade): {expectancy:+.3f}R")
    print(f"\n  Total realized $:      ${total_realized:+,.2f}")

## 5. Breakdown by conviction

Does `High` conviction actually win more often than `Medium`? If not, the conviction bucketing isn't doing useful work and Alex should re-examine the scoring.

In [ ]:
if not replay_df.empty and "conviction" in replay_df.columns:
    closed = replay_df[replay_df["status"].isin(["stop", "target"])].copy()
    if not closed.empty:
        closed["won"] = closed["status"] == "target"
        by_conv = closed.groupby("conviction").agg(
            n=("won", "size"),
            wins=("won", "sum"),
            win_rate=("won", "mean"),
            avg_realized=("realized", "mean"),
        ).reset_index()
        by_conv
    else:
        print("No closed trades to break down by conviction.")
else:
    print("No conviction column or empty replay.")

### What to look for

- **High wins > Medium wins**: conviction is doing useful sorting.
- **High wins ≈ Medium wins**: conviction is noise. Alex should weight conviction less in his sizing.
- **High wins < Medium wins**: 🚩 the conviction signal is INVERTED — the engine's most-confident calls are losing more than its less-confident ones. This is a known failure mode in over-fit confluence systems and is one of the failure modes notebook-04's `validate` is designed to catch.

## 6. The journal entry

End every audit with a one-block markdown entry suitable for pasting into Alex's trading journal. The format is opinionated; adapt it for your own system.

In [ ]:
if replay_df.empty:
    print("Nothing to journal.")
else:
    closed = replay_df[replay_df["status"].isin(["stop", "target"])]
    wins = closed[closed["status"] == "target"]
    losses = closed[closed["status"] == "stop"]
    win_rate = len(wins) / len(closed) if len(closed) else 0.0
    total_realized = replay_df["realized"].sum()

    journal = f"""
## Trading journal — {date.today().isoformat()}

**Workbook audited**: `{WORKBOOK.name}`
**Plans in workbook**: {len(replay_df)}  ({len(closed)} closed, {(replay_df['status'] == 'open').sum()} still open)

### Results

- **Win rate**: {win_rate:.1%}  ({len(wins)}W / {len(losses)}L)
- **Realized $ on closed trades**: ${total_realized:+,.2f}

### Top winners (by realized $)

{wins.nlargest(3, 'realized')[['symbol', 'realized', 'bars_held']].to_string(index=False) if len(wins) else '(none)'}

### Top losers (by realized $)

{losses.nsmallest(3, 'realized')[['symbol', 'realized', 'bars_held']].to_string(index=False) if len(losses) else '(none)'}

### Discipline notes

_(Fill in: which trades did I take vs skip? Where did I deviate from the recommendation? Why?)_
"""
    print(journal)

## 7. The deviation log (manual)

The engine cannot know what trades Alex *actually* took. To complete the audit, Alex needs to match the recommendations against his broker's trade history and flag deviations. Two categories:

- **Took but engine said skip**: a trade with no corresponding `High`-conviction plan in the workbook. Often discretionary; sometimes hindsight bias.
- **Engine said take but skipped**: a `High`-conviction plan in the workbook that Alex didn't execute. Usually risk-aversion in the moment; sometimes saved him money, sometimes cost him.

Track both. The pattern over weeks reveals whether Alex is **over-trading** (lots of took-but-engine-said-skip) or **under-trading** (lots of engine-said-take-but-skipped) — both are repairable, but only if you measure them.

In [ ]:
# Template — fill in from your broker's trade history:
deviations = pd.DataFrame([
    # {"symbol": "MSFT", "action": "took", "engine_said": "skip", "reason": "chart looked good to me; engine score was 0.32"},
    # {"symbol": "NVDA", "action": "skipped", "engine_said": "BUY-High", "reason": "already up 5% intraday; chased fear"},
])

if deviations.empty:
    print("No deviations logged. Fill in `deviations` above by hand from your broker.")
else:
    deviations

## 8. The closing question

Every disciplined trader's evening should end with this:

> *"If I had followed the engine's recommendations exactly — every `High` conviction at full size, every `Medium` at half size, skipped everything else — would today have been better or worse than what I actually did?"*

Two outcomes:

1. **"Better"** — Alex deviated and it hurt him. The discipline gap is real. Tighten the rules; perhaps automate the entries he keeps skipping.
2. **"Worse"** — Alex's discretion added value. Either he's the rare trader whose intuition genuinely beats the system (unlikely but possible) OR he got lucky today and the random walk happened to favor his deviations. Wait for a 30-day sample before drawing conclusions.

Most days, the honest answer is **"about the same."** That's fine. The discipline isn't about beating the engine; it's about staying in the game long enough for the edge to compound.

---

## Series complete

You've reached the end of *A Developer Guide to Disciplined Trading*. Six notebooks:

| # | Notebook | What it shows |
|---|---|---|
| 01 | foundations | The full breadth of `obb.techtrade.*` |
| 02 | morning-scan | Daily morning routine: scan → filter → export |
| 03 | single-position-deep-dive | One ticker through plan → orders → simulate |
| 04 | validation-gate | `validate` with PBO + DSR + verdict |
| 05 | tuning-per-sector | `tune` and the persist-and-auto-load chain |
| 06 | audit-and-replay | End-of-day discipline journal |

When `narrator` (#84) or `MCP tools` (#85) ship, notebook 07 will join the series.

---

*End of notebook 06 and end of the series. Maintained on the `trading_technicals` branch of `prajoria/OpenBB`. Issues, contributions, and corrections welcome.*